In [1]:
import re
import cobra
import numpy as np
from cobra.io import read_sbml_model, write_sbml_model
from cobra.util.array import create_stoichiometric_matrix

In [2]:
# Read model

#E coli minimal model
model_name = 'ecoli5010_no_b'

#M model
#model_name = 'M_model

#PQS model
#model_name = 'PQS_model'


In [3]:
model = read_sbml_model('../models/' + model_name + ".xml")

In [4]:
# Create stoichiometric matrix
S = create_stoichiometric_matrix(model)

In [5]:
# Transpose S matrix
D_S = S.transpose()

In [6]:
# Dual identity matrix (v's) and concatenate to D_S
I = np.identity(len(D_S))
D_S = np.concatenate([D_S, I], axis=1)

In [7]:
# Make irreversible matrix from irreversible reactions (z's)
irreversibility = [not rxn.reversibility for rxn in model.reactions]
irreversibility_mat = np.diag(irreversibility).astype(int) * -1
irreversibility_mat = irreversibility_mat[:, irreversibility]

In [8]:
# Concatenate D_S with z's
D_S = np.concatenate([D_S, irreversibility_mat], axis=1)

In [9]:
# Define target reaction and get index

#E coli minimal model target
target_rxn = "Biomass_Ecoli_core_w_GAM"

#PQS model target
#model_name = 'R04

#M model target
#model_name = 'r5'

In [10]:
target = model.reactions.index(target_rxn)

In [11]:
# Secrete target metabolites (w)
# model.reactions.index(target)
w = np.zeros(len(D_S), dtype="int")
w[target] = -1

In [12]:
# Concatenate D_S with w
D_S = np.column_stack((D_S, w))
np.shape(D_S)

(68, 182)

In [13]:
# Decide for each reaction if it is reversible (list of indices, not binary)
rev_u = list(np.ones(len(model.metabolites), dtype="int"))
rev_v = [0 if irrev == True else 1 for irrev in (irreversibility)]
rev_z = list(np.zeros(np.shape(irreversibility_mat), dtype="int")[1])
rev_w = [0]
D_rev_rxns = rev_u + rev_v + rev_z + rev_w

In [14]:
# Reaction names
rxn_u = [metabolite.id for metabolite in model.metabolites]
rxn_v = [f"v{x+1}" for x in range(0, len(model.reactions))]
rxn_z = [f"z{x+1}" for x in range(0, len(model.reactions)) if irreversibility[x] == 1]
reaction_names = rxn_u + rxn_v + rxn_z + ["w"]

In [15]:
# Give metabolite IDs
metab_names = [reaction.id for reaction in model.reactions]

In [16]:
print(D_S.shape)
print(len(D_rev_rxns))
print(len(reaction_names))
print(len(metab_names))

(68, 182)
182
182
68


In [17]:
# Clean metabolite names from characters

import re


def clean_name(name):
    # Remove parentheses, slashes, plus, minus, & symbols, etc.
    name = re.sub(r"[^\w\d_-]", "_", name)  # only letters, digits, _ and - allowed
    # optionally truncate to max 50 chars
    return name[:50]


# Clean reaction names
reaction_names_clean = [clean_name(r) for r in reaction_names]

# Clean metabolite names
metab_names_clean = [clean_name(m) for m in metab_names]

In [18]:
# Dual S matrix for ECM (Extracellular metabolite for reactions w)

#V metabolites
v_u = np.zeros((len(model.reactions), len(model.metabolites)), dtype=int)
v_I = -I
v_z = np.zeros(irreversibility_mat.shape, dtype=int)
v_w = np.zeros((len(model.reactions),1), dtype=int)

#Z metabolites

MZ = np.concatenate([
    np.zeros(len(model.metabolites)),
    np.zeros(len(model.reactions)),
    np.ones(sum(irreversibility)),
    np.zeros(1)
])

#MW metabolite
Mw = np.zeros(D_S.shape[1], dtype=int)
Mw[-1] = 1

In [19]:
# Dual S matrix for ECM

#Submatrix of v mets
v_met = np.concatenate([v_u,v_I,v_z,v_w], axis = 1)
#Join
D_met = np.vstack((v_met, MZ))
#Add MW
D_met = np.vstack((D_met, Mw))

ECM_D_S = np.concatenate([D_S,D_met], axis = 0)

In [20]:
ECM_D_S.shape

(138, 182)

In [21]:
#Decide for each reaction if it is reversible
ECM_D_rev_rxns = [x for x in range(len(D_rev_rxns)) if D_rev_rxns[x] == 1]
#ECM_D_rev_rxns = [x for x in range (len(model.reactions)+len(model.metabolites))]

In [22]:
# Decide which metabolites are external (will not be held in steady-state, and will occur in ECMs)
external = list(range(len(model.reactions), ECM_D_S.shape[0]))

In [23]:
met_w = "MW"
v_met_names = ["V"+str(i+1) for i in range(I.shape[1])]
z_met_names = ['MZ']

ECM_metab_names = metab_names_clean + v_met_names + z_met_names + [met_w]

In [24]:
# --------------------
# 2. Create model
# --------------------
D_model = cobra.Model("D_model")

# --------------------
# 3. Add metabolites
# --------------------
mets = {}  # dictionary of all metabolites

for i, name in enumerate(ECM_metab_names):
    if i in external:
        mets[name] = cobra.Metabolite(id=name, name=name, compartment="e")
    else:
        mets[name] = cobra.Metabolite(id=name, name=name, compartment="c")
# --------------------
# 4. Add reactions with stoichiometry from S matrix
# --------------------
for j, rxn_name in enumerate(reaction_names_clean):
    rxn = cobra.Reaction(rxn_name)
    if j in ECM_D_rev_rxns:
        rxn.lower_bound = -1000
    else:
        rxn.lower_bound = 0  # irreversible
    rxn.upper_bound = 1000
    # stoichiometry: S[i,j] is metabolite i in reaction j
    stoich = {ECM_metab_names[i]: ECM_D_S[i, j] for i in range(len(ECM_metab_names))}
    rxn.add_metabolites({mets[m]: coeff for m, coeff in stoich.items()})

    D_model.add_reactions([rxn])

#Add exchange reactions to external metabolites
for i in external:
    ex_id = f"EX_{ECM_metab_names[i]}"
    ex_rxn = cobra.Reaction(ex_id)
    
    # Allow import and export
    ex_rxn.lower_bound = -1000
    ex_rxn.upper_bound = 1000
    
    ex_rxn.add_metabolites({D_model.metabolites.get_by_id(ECM_metab_names[i]): -1})
    D_model.add_reactions([ex_rxn])

D_model.objective = D_model.reactions.get_by_id("w")
# --------------------
# 5. Done
# --------------------
print(D_model.summary())

Objective
1.0 w = 1000.0

Uptake
------
Metabolite Reaction  Flux  C-Number C-Flux
       V55   EX_V55 79.99         0  0.00%

Secretion
---------
Metabolite Reaction  Flux  C-Number C-Flux
        MW    EX_MW -1000         0  0.00%



In [25]:
D_model

Name,D_model
Memory address,7f67b402fb50
Number of metabolites,138
Number of reactions,252
Number of genes,0
Number of groups,0
Objective expression,1.0*w - 1.0*w_reverse_f1290
Compartments,"c, e"


In [26]:
dual_model_path = "dual_MZ_" + model_name 
write_sbml_model(D_model, "../models/" + dual_model_path + ".xml")